# 03 — Meta-Learners (S/T/X/R + Uplift Tree)
Fit causalml meta-learners on the observational simulation.

In [ ]:
import sys
sys.path.insert(0, "..")
from src.meta_learners import run_all_meta_learners
from src.evaluation import evaluate_all, get_uplift_curve
from src.viz import plot_uplift_curves, plot_cate_distribution, plot_shap_beeswarm
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_parquet("../data/simulation_observational.parquet")
meta_cols = [c for c in df.columns if c not in ["treatment", "outcome", "tau_true", "propensity"]]
print(f"Feature columns: {len(meta_cols)}")
df.shape

## Run all meta-learners

In [ ]:
results = run_all_meta_learners(
    df,
    feature_cols=meta_cols,
    n_bootstrap=50,
    run_uplift_tree=True,
)
tau_hats = {name: res["tau_hat"] for name, res in results.items()}
for name, v in tau_hats.items():
    print(f"{name}: mean tau = {v.mean():.4f}")

## Evaluation metrics

In [ ]:
metrics = evaluate_all(df, tau_hats)
print(metrics.to_string())

## Uplift curves

In [ ]:
uplift_dfs = {
    name: get_uplift_curve(tau, df["treatment"].values, df["outcome"].values)
    for name, tau in tau_hats.items()
}
fig = plot_uplift_curves(uplift_dfs, save=True)
fig.show()

## CATE distribution by recency

In [ ]:
subset_tau = {k: v for k, v in tau_hats.items() if k in ["S", "T", "X", "R"]}
fig = plot_cate_distribution(df, tau_hats=subset_tau, save=True)
fig.show()

## SHAP beeswarm for X-learner

In [ ]:
if results["X"]["shap_values"] is not None:
    fig = plot_shap_beeswarm(results["X"]["shap_values"], df[meta_cols], model_name="X-learner", save=True)
    fig.show()
else:
    print("SHAP values not available for X-learner.")

In [ ]:
import pickle
with open("../results/meta_learner_results.pkl", "wb") as f:
    pickle.dump(results, f)
metrics.to_csv("../results/meta_learner_metrics.csv")
print("Saved.")